In [1]:
from faster_whisper import WhisperModel

model = WhisperModel("small", device="cpu", compute_type="int8")

video_path = "../data/videos/milestone1.mp4"

segments, info = model.transcribe(video_path, language="en")

transcript = ""
for segment in segments:
    transcript += segment.text.strip() + " "

with open("transcript.txt", "w", encoding="utf-8") as f:
    f.write(transcript)

print(transcript[:1000])

Hello everyone, my name is Thao. And today I will present my projects, which is transformer based video transfer summarization focused on mid-week meeting. First of all, let's talk a little bit about the problems in the modern organization nowadays. Meeting takes a lot of time for everyone, including the employee and the manager. So because meetings are long and information dense, important insights often become buried in silent transcripts. After the meeting, usually the team have to create the meeting minutes or summaries. However, manual summarization has a lot of challenges. It can be time consuming, the amount of information can be overwhelming, summaries can be inconsistent between people, and sometimes the important insight may be missed. So these challenges motivate me to implement the automated meeting summarization systems. The goal of this project is to view the system that can automatically generate summaries from videos with a focus on meetings and presentations. The idea 

In [2]:
import re
import time
from transformers import BartTokenizerFast, BartForConditionalGeneration

# ========= 1) Load model =========
MODEL_DIR = "../outputs/qmsum_bart_baseline/best_model"

print("Loading tokenizer...")
tokenizer = BartTokenizerFast.from_pretrained(MODEL_DIR)

print("Loading model...")
model = BartForConditionalGeneration.from_pretrained(MODEL_DIR)

print("Model loaded successfully.\n")


# ========= 2) Inputs =========
# query = "What are the main findings and conclusions of this presentation?"
# section_query = (
#     "What key ideas or findings are discussed in this part of the lecture? "
#     "Write a concise summary of the content."
# )
section_query = "Write a concise summary of the important points in this section."

# ========= 3) Helpers =========

def clean_text(text):
    text = text.replace(">>", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text


def split_into_sentence_chunks(text, tokenizer, max_tokens=320):
    print("Cleaning transcript...")
    text = clean_text(text)

    sentences = re.split(r'(?<=[.!?])\s+', text)
    sentences = [s.strip() for s in sentences if s.strip()]

    print(f"Total sentences detected: {len(sentences)}")

    chunks = []
    current = []

    for sent in sentences:
        candidate = " ".join(current + [sent])
        token_count = len(tokenizer.encode(candidate, add_special_tokens=False))

        if token_count > max_tokens and current:
            chunks.append(" ".join(current))
            current = [sent]
        else:
            current.append(sent)

    if current:
        chunks.append(" ".join(current))

    print(f"Total chunks created: {len(chunks)}\n")

    for i,c in enumerate(chunks):
        tokens = len(tokenizer.encode(c))
        print(f"Chunk {i+1} token length: {tokens}")

    print()
    return chunks


def generate_summary(query, text, tokenizer, model,
                     input_max_length=512,
                     output_max_length=120,
                     output_min_length=40):

    start_time = time.time()

    input_text = f"query: {query}\n\ntranscript:\n{text}"

    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        max_length=input_max_length,
        truncation=True
    )

    input_tokens = inputs["input_ids"].shape[1]

    print(f"Input tokens: {input_tokens}")

    summary_ids = model.generate(
        **inputs,
        max_length=output_max_length,
        min_length=output_min_length,
        num_beams=4,
        no_repeat_ngram_size=3,
        length_penalty=1.0,
        early_stopping=True
    )

    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True).strip()

    generation_time = round(time.time() - start_time, 2)

    print(f"Generated tokens: {len(summary_ids[0])}")
    print(f"Generation time: {generation_time}s\n")

    # Cut incomplete sentence
    if not summary.endswith((".", "!", "?")):
        last_punc = max(summary.rfind("."), summary.rfind("!"), summary.rfind("?"))
        if last_punc > 40:
            summary = summary[:last_punc + 1]

    return summary


# ========= 4) Pipeline =========

print("Splitting transcript into chunks...\n")

chunks = split_into_sentence_chunks(transcript, tokenizer)

chunk_summaries = []

print("Starting chunk summarization...\n")

for i, chunk in enumerate(chunks):

    print("="*60)
    print(f"Processing chunk {i+1}/{len(chunks)}")

    summary = generate_summary(
        query=section_query,
        text=chunk,
        tokenizer=tokenizer,
        model=model,
        output_max_length=100,
        output_min_length=30
    )

    print("Chunk summary:")
    print(summary)
    print()

    chunk_summaries.append(summary)


# ========= 5) Final summary =========

print("\n" + "="*80)
print("Combining chunk summaries for final summarization...\n")

combined_text = "\n".join(
    [f"Section {i+1}: {s}" for i,s in enumerate(chunk_summaries)]
)

def generate_long_summary(text, tokenizer, model):
        prompt = f"""
    Rewrite the following lecture summary so that it is slightly shorter and clearer.
    Keep all important ideas and preserve the structure of the sections.
    Do not remove major findings or conclusions.

    {text}
    """

        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            max_length=1024,
            truncation=True
        )

        summary_ids = model.generate(
            **inputs,
            max_length=450,      # much longer
            min_length=250,      # force long output
            num_beams=4,
            length_penalty=1.0,
            no_repeat_ngram_size=3,
            early_stopping=True
        )

        return tokenizer.decode(summary_ids[0], skip_special_tokens=True)

final_summary = generate_summary(
    query="What are the main findings, methods, and conclusions of the full lecture?",
    text=combined_text,
    tokenizer=tokenizer,
    model=model,
    output_max_length=5000,
    output_min_length=250
)
# final_summary = generate_long_summary(
#     combined_text,
#     tokenizer,
#     model
# )

print(final_summary)
print("="*80)
print("FINAL SUMMARY\n")
print(final_summary)

print("\nPipeline finished successfully.")

Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading tokenizer...
Loading model...


Loading weights:   0%|          | 0/358 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Model loaded successfully.

Splitting transcript into chunks...

Cleaning transcript...
Total sentences detected: 51
Total chunks created: 5

Chunk 1 token length: 306
Chunk 2 token length: 298
Chunk 3 token length: 311
Chunk 4 token length: 315
Chunk 5 token length: 19

Starting chunk summarization...

Processing chunk 1/5
Input tokens: 326
Generated tokens: 66
Generation time: 6.93s

Chunk summary:
According to Thao, it was hard to have a system that can automatically generate summaries from videos with a focus on meetings and presentations.

Processing chunk 2/5
Input tokens: 318
Generated tokens: 64
Generation time: 4.6s

Chunk summary:
For this project about the dataset, I use Q&Zoom, which is query-based meeting summarization. Each meeting example contains three main components, meeting transcript, vary, and reference summary. The goal of the model is to generate a summary that is relevant to both the transcript and the vary.

Processing chunk 3/5
Input tokens: 331
Generated toke

In [4]:
from IPython.display import display, Markdown

display(Markdown(combined_text))

Section 1: According to Thao, it was hard to have a system that can automatically generate summaries from videos with a focus on meetings and presentations.
Section 2: For this project about the dataset, I use Q&Zoom, which is query-based meeting summarization. Each meeting example contains three main components, meeting transcript, vary, and reference summary. The goal of the model is to generate a summary that is relevant to both the transcript and the vary.
Section 3: The decoder is autoregressive, which means that it generates the summary when talking at a time based on encoded representation of the input. The interesting part is that part first pre-train using the denoising objective.
Section 4: The training loss over the training steps, the step is around 4,000 steps, and we can see that the loss decreases from around 4.0 to about 3.5, which means that the model is learning to generate the summary that is better match the reference summaries.
Section 5: The main point of the project was that the song one was about to be released. The main point was that it should be about how they should be able to release the new material. The new material was about how to make it into a new song one.